# Day 2 notebook companion

Run in order with synthetic data. Mermaid diagrams render on the website. Setup and shared helpers are embedded; no checkout is required. Learner exercises report NOT ATTEMPTED until implemented. Reference checks are separate. Optional controls also have direct function calls.


In [ ]:
import importlib.metadata
import subprocess
import sys
for package, version in {"cryptography": "50.0.1", "matplotlib": "3.10.6", "ipywidgets": "8.1.7"}.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])
print("Dependencies ready. Restart if an older library was already imported, then run all cells.")


## Shared teaching helpers

Inspect this implementation. TLS uses real SSL objects over memory buffers and temporary test key files; no system trust changes or network listeners. The teaching KDF is not a standardized protocol key schedule.


In [ ]:
"""Day 2 teaching helpers. Real TLS over MemoryBIO; no sockets or trust-store changes."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import ssl
import tempfile
import hashlib
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtendedKeyUsageOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


def make_pki(expired=False):
    """Create an isolated root, intermediate, server, and client for this run."""
    now = datetime.now(timezone.utc)
    keys = {name: ec.generate_private_key(ec.SECP256R1())
            for name in ('root', 'intermediate', 'server', 'client')}
    names = {name: x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Workshop ' + name)])
             for name in keys}
    certs = {}
    for name, issuer, ca, path_length, eku in [
        ('root', 'root', True, 1, None),
        ('intermediate', 'root', True, 0, None),
        ('server', 'intermediate', False, None, ExtendedKeyUsageOID.SERVER_AUTH),
        ('client', 'intermediate', False, None, ExtendedKeyUsageOID.CLIENT_AUTH),
    ]:
        end = now - timedelta(days=1) if expired and name == 'server' else now + timedelta(days=7)
        builder = (x509.CertificateBuilder().subject_name(names[name]).issuer_name(names[issuer])
                   .public_key(keys[name].public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=2)).not_valid_after(end)
                   .add_extension(x509.BasicConstraints(ca=ca, path_length=path_length), critical=True)
                   .add_extension(x509.KeyUsage(digital_signature=True, content_commitment=False,
                       key_encipherment=False, data_encipherment=False, key_agreement=False,
                       key_cert_sign=ca, crl_sign=ca, encipher_only=False, decipher_only=False), critical=True)
                   .add_extension(x509.SubjectKeyIdentifier.from_public_key(keys[name].public_key()), False)
                   .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(keys[issuer].public_key()), False))
        if eku:
            builder = builder.add_extension(x509.ExtendedKeyUsage([eku]), False)
            builder = builder.add_extension(x509.SubjectAlternativeName([
                x509.DNSName('invoice.test' if name == 'server' else 'client.test')]), False)
        certs[name] = builder.sign(keys[issuer], hashes.SHA256())
    return keys, certs


def tls_trial(hostname='invoice.test', trust_root=True, expired=False,
              include_intermediate=True, mtls=False, send_client=True,
              client_wrong_eku=False):
    """Handshake and exchange application bytes. Failures raise ssl.SSLError.

    Private PEM files are disposable teaching keys in a temporary directory.
    Does not implement online revocation, networking, or authorization policy.
    """
    keys, certs = make_pki(expired)
    pem = lambda c: c.public_bytes(serialization.Encoding.PEM)
    with tempfile.TemporaryDirectory(prefix='workshop-pki-') as directory:
        base = Path(directory)
        for name in ('server', 'client'):
            selected = 'server' if name == 'client' and client_wrong_eku else name
            chain = pem(certs[selected])
            if include_intermediate or name == 'client':
                chain += pem(certs['intermediate'])
            (base / (name + '.pem')).write_bytes(chain)
            (base / (name + '.key')).write_bytes(keys[selected].private_bytes(
                serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                serialization.NoEncryption()))
        server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
        client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        for context in (server_context, client_context):
            context.minimum_version = context.maximum_version = ssl.TLSVersion.TLSv1_3
        server_context.load_cert_chain(str(base / 'server.pem'), str(base / 'server.key'))
        if trust_root:
            client_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if mtls:
            server_context.verify_mode = ssl.CERT_REQUIRED
            server_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if send_client:
            client_context.load_cert_chain(str(base / 'client.pem'), str(base / 'client.key'))
        ci, co, si, so = (ssl.MemoryBIO() for _ in range(4))
        client = client_context.wrap_bio(ci, co, server_hostname=hostname)
        server = server_context.wrap_bio(si, so, server_side=True)
        completed = [False, False]

        def transfer():
            for outgoing, incoming in ((co, si), (so, ci)):
                if outgoing.pending:
                    incoming.write(outgoing.read())

        for _ in range(100):
            for index, peer in enumerate((client, server)):
                if not completed[index]:
                    try:
                        peer.do_handshake()
                        completed[index] = True
                    except ssl.SSLWantReadError:
                        pass
            transfer()
            if all(completed):
                break
        else:
            raise RuntimeError('TLS handshake stalled')
        payload = b'synthetic confidential invoice'
        client.write(payload)
        transfer()
        assert server.read(4096) == payload
        return {'version': client.version(), 'cipher': client.cipher()[0],
                'client_authenticated': bool(server.getpeercert()),
                'application_bytes': len(payload)}


def expect_rejection(operation, exceptions):
    """Assert the negative case, without accepting a silent failure."""
    try:
        operation()
    except exceptions:
        return
    raise AssertionError('Expected rejection did not occur')


def derive_day2(secret, transcript, direction=b'alice-to-bob'):
    """Teaching KDF only, not a standardized TLS or hybrid key schedule."""
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                info=b'workshop-day2:v1|' + hashlib.sha256(transcript).digest()
                + b'|' + direction).derive(secret)


# Session 10: Hybrid Key Establishment

**45 minutes taught · 75–90 minutes independently.** Instructor (see course website) · Notebook (see course website)

## Outcomes and setup

Explain why a transition may combine classical and PQ contributions; demonstrate that context and both input secrets affect a teaching KDF; and identify downgrade and interoperability risks. Complete Sessions 4 and 9. Use the Day 2 setup (see course website).



## Why combine mechanisms?

A migration must balance exposure to quantum cryptanalysis with implementation and ecosystem uncertainty. A carefully designed hybrid construction aims to retain useful security if one component's assumptions fail. That aim depends on the specific combiner, protocol, authentication and security model. “Two algorithms” is not a proof of “at least one always protects us.”

```mermaid
flowchart TD
    X["X25519 shared contribution"] --> C["Specified combiner and key schedule"]
    K["ML-KEM shared contribution"] --> C
    T["Authenticated transcript, roles and negotiated suite"] --> C
    C --> A["Traffic keys for AEAD"]
```

Read all three inputs. If the attacker can choose a classical-only mode without an authenticated policy decision, the intended combined protection may disappear. If authentication remains classically vulnerable, PQ key establishment alone does not make every active-attack property post-quantum.

## Experiment: explicit inputs and domain separation

The code below is a **teaching combiner, not a standardized TLS group or deployable handshake**. It joins two fixed 32-byte secrets before HKDF and binds a transcript. This illustrates dependencies, not a general robust-combiner proof. Deploy a specified, reviewed protocol profile instead of adopting this example's labels or encoding.


In [ ]:
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
from cryptography.hazmat.primitives.asymmetric.mlkem import MLKEM768PrivateKey
alice_x, bob_x = X25519PrivateKey.generate(), X25519PrivateKey.generate()
zx_a = alice_x.exchange(bob_x.public_key())
zx_b = bob_x.exchange(alice_x.public_key())
bob_pq = MLKEM768PrivateKey.generate()
zp_a, ct = bob_pq.public_key().encapsulate()
zp_b = bob_pq.decapsulate(ct)
suite = b'TEACHING-ONLY:X25519+ML-KEM-768:v1'
transcript = (suite + alice_x.public_key().public_bytes_raw()
              + bob_x.public_key().public_bytes_raw()
              + bob_pq.public_key().public_bytes_raw() + ct)
def teaching_combiner(classical, pq, context):
    if len(classical) != 32 or len(pq) != 32:
        raise ValueError('Both fixed-length contributions are required')
    return derive_day2(classical + pq, context)
key_a = teaching_combiner(zx_a, zp_a, transcript)
key_b = teaching_combiner(zx_b, zp_b, transcript)
assert key_a == key_b
assert key_a != teaching_combiner(zx_a, bytes(32), transcript)
assert key_a != teaching_combiner(bytes(32), zp_a, transcript)
assert key_a != teaching_combiner(zx_a, zp_a, transcript + b'changed')
expect_rejection(lambda: teaching_combiner(zx_a, b'', transcript), ValueError)
print('PASS: teaching combination depends on both secrets and exact context')


Changing one input and observing different output is only a functional check. Replacing an input with zeros here models a sensitivity experiment, not a production fallback or a proof of resistance to an adaptive attacker.

## Downgrade is a policy and transcript problem

```mermaid
sequenceDiagram
    participant C as Client policy requires hybrid
    participant M as Active attacker
    participant S as Server
    C->>M: Offered capabilities
    M->>S: Tries to strip PQ capability
    S->>C: Selected mode in authenticated handshake
    C->>C: Check transcript and local minimum policy
    Note over C,S: Reject incompatible policy rather than silently weaken it
```

The receiver needs both an authenticated account of negotiation and an independent minimum policy. A list of permitted names in Python does not authenticate negotiation; it only illustrates local policy after authentication.


In [ ]:
def require_hybrid(selected, authenticated, required=True):
    if not authenticated:
        raise ValueError('Unauthenticated negotiation')
    if required and selected != 'approved-hybrid-profile':
        raise ValueError('Local minimum policy not met')
    return selected
assert require_hybrid('approved-hybrid-profile', True) == 'approved-hybrid-profile'
expect_rejection(lambda: require_hybrid('classical-only', True), ValueError)
expect_rejection(lambda: require_hybrid('approved-hybrid-profile', False), ValueError)
print('PASS: minimum policy rejects downgrade and unauthenticated selection')


## Engineer the migration

| Decision | Evidence to collect |
| --- | --- |
| Exact profile and version | Agreed specification, code points, parameters and implementation versions |
| Authentication | Which signatures and trust chains authenticate the exchange? |
| Interoperability | Client/server matrix, proxies, middleboxes, certificate tooling and failure behavior |
| Resource use | Full handshake bytes, latency distributions, memory and failure-load behavior |
| Rollout | Inventory owners, negotiated-mode telemetry, staged deployment and documented minimum policy |

Do not advertise a service as hybrid because a library exposes both primitives. Observe what the actual connection negotiated. Separate negotiation failures from ordinary network errors. A rollback plan must say who can approve reduced protection and for which data; automatic fallback on attacker-induced errors is dangerous.

A classical-only certificate path and a hybrid KEM address different assumptions. During transition, document which protection applies to passive recordings, active impersonation, and future software updates. A powerful state actor can exploit a forgotten fallback, a legacy intermediary or an unauthorized endpoint regardless of the combiner.

## Practice and answers

An archive client requires hybrid protection, but a legacy proxy only accepts classical key exchange. Propose a delivery decision without silently relaxing the archive policy. Then explain why two separately encrypted copies of a secret are not automatically a robust combiner.

<details><summary>Worked answers</summary>
<p>Identify the incompatible termination hop, upgrade or replace it, and fail the required-protection connection until the approved profile works. If an exception is necessary, it is an explicit data-owner risk decision, not an automatic crypto fallback. Combining mechanisms requires analysis of what an attacker learns from each component and how the final secret is derived; two ciphertexts can expose the secret if either independently reveals it.</p>
</details>

Exit check: explain the difference between demonstrating that two inputs affect a KDF and proving a protocol remains secure when one component fails. Continue to PQ signatures (see course website).

## Sources and limits

Reviewed 22 September 2026: [FIPS 203](https://csrc.nist.gov/pubs/fips/203/final), [TLS 1.3](https://www.rfc-editor.org/rfc/rfc8446), [HKDF RFC 5869](https://www.rfc-editor.org/rfc/rfc5869). No current IETF hybrid deployment profile is implemented or claimed by this notebook.


In [ ]:
print("PASS: completed session-10-hybrid-key-establishment demonstrations; learner status is reported separately")
